In [1]:
import pandas as pd
from tqdm.notebook import tqdm

## Setup

Load the raw roll call vote data (assuming it is clean - it is not currently but lets just pretend for now), list of bills voted on and the list of all representatives.

We only care about the list of bills because that gives us access to the general court number the vote took place which can help us when joining these tables.

In [2]:
bill_df = pd.read_csv('../data/parsedcsv/hjbills_raw.csv')
vote_df = pd.read_csv('../data/parsedcsv/hjrollcallvotes_raw.csv')
rep_df = pd.read_csv('../data/parsedcsv/MAlegislators_GC182-194.csv')

In [3]:
bill_df.head()

,court_number,rollcall_count_for_year,bill_name,rollcall_date,rollcall_time,yea_vote_count,nay_vote_count,not_voting_count,validated
0,183,1,QUORUM,1/8/2003,1:44 PM,151,0,9,True
1,183,2,H2003 On motion to Postpone,1/8/2003,2:01 PM,45,107,8,True
2,183,3,H2003 On Amendment,1/8/2003,2:20 PM,41,110,9,True
3,183,4,H2003 On Amendment,1/8/2003,2:45 PM,44,107,9,True
4,183,5,H2003 On Amendment,1/8/2003,4:03 PM,28,119,13,True


In [4]:
vote_df.head()

,court_number,rollcall_count_for_year,voter_name,vote_value
0,183,1,Mr. Speaker,Y
1,183,1,Fresolo,Y
2,183,1,"Murphy, K.J.",Y
3,183,1,Swan,Y
4,183,1,Asselin,Y


In [5]:
len(vote_df)

863805

Separate the rep_df into house and senate representatives for later use.

In [7]:
rep_df
rep_df.drop(inplace=True, columns=['Unnamed: 0'])

In [8]:
house_rep_df = rep_df[rep_df['branch'] == 'House']
senate_rep_df = rep_df[rep_df['branch'] == 'Senate']

house_rep_df.to_csv('MAHouseLegislators_GC182-194.csv')
senate_rep_df.to_csv('MASenateLegislators_GC182-194.csv')

I filter down the DF of representatives to only those that serve in the House since we are not dealing with the Senate at all right now

In [9]:
rep_df = rep_df[rep_df['branch'] == 'House']
rep_df.head()

,full_name,member_code,branch,party_affiliation,district,court_year,court_number
0,Marie P. St. Fleur,MPS1,House,Democrat,NaN,2001,182
3,Mary S. Rogeness,MSR1,House,Republican,NaN,2001,182
4,Mark V. Falzone,MVF1,House,Democrat,NaN,2001,182
6,Nancy Flavin,N F1,House,Democrat,NaN,2001,182
7,Nancy J. Caffyn,NJC1,House,Republican,NaN,2001,182


Checking whether our list of representatives covers the years of votes we have, we see that the list of representatives actually goes beyond the years we have of votes (2002 does appear in the Roll Call website, but does not have any roll call votes PDFs saved for it, so technically they should match but no data exists for 2002 despite it having a page for it.)

This means we should be covered and can successfully merge our rep dataset for all years of roll call votes.

In [10]:
bill_df.court_number.unique()

array([183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194],
      dtype=int64)

In [11]:
rep_df.court_number.unique()

array([182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194],
      dtype=int64)

### Minor data cleaning

We need to replace all instances of Mr. Speaker with the standing speaker of the house at each instance. This method is currently overgeneralized (Speakers rarely transitioned at the change of a GC), but for quick prototyping, I am making the following assumptions for now.

< 183 = Finneran (until 09/28/2004) = until court 184 rollcall_count_for_year 1

< 186 = DiMasi (until 01/27/2009) = until court 186 rollcall_count_for_year 232

< 191 = DeLeo (until 12/29/2020) = until court 192

< 194 = Mariano (until present) = until present

In [12]:
vote_df['is_speaker'] = vote_df['voter_name'].str.contains('Speaker')

In [13]:
# Create a mask for rows containing "Speaker" in voter_name
speaker_mask = vote_df['voter_name'].str.contains('Speaker', case=False, na=False)

# Apply replacements based on general court number and roll call vote number
# Court 183 (until Court 184 rollcall 1) = Finneran
vote_df.loc[speaker_mask & (vote_df['court_number'] < 184), 'voter_name'] = 'Finneran'

# Court 184 rollcall 1 through Court 186 rollcall 232 = DiMasi
vote_df.loc[speaker_mask & (vote_df['court_number'] == 184), 'voter_name'] = 'DiMasi'
vote_df.loc[speaker_mask & (vote_df['court_number'] == 185), 'voter_name'] = 'DiMasi'
vote_df.loc[speaker_mask & (vote_df['court_number'] == 186) & (vote_df['rollcall_count_for_year'] <= 232), 'voter_name'] = 'DiMasi'

# After Court 186 rollcall 232 through Court 191 (until Court 192) = DeLeo
vote_df.loc[speaker_mask & (vote_df['court_number'] == 186) & (vote_df['rollcall_count_for_year'] > 232), 'voter_name'] = 'DeLeo'
vote_df.loc[speaker_mask & (vote_df['court_number'].isin([187, 188, 189, 190, 191])), 'voter_name'] = 'DeLeo'

# Court 192 onwards (Court 194 and present) = Mariano
vote_df.loc[speaker_mask & (vote_df['court_number'] >= 192), 'voter_name'] = 'Mariano'

In [14]:
vote_df[vote_df['voter_name'].str.contains('Speaker') == True]

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker


Next we want to check in case there are any votes that missing a name. Fortunately, that is not the case.

In [15]:
vote_df[vote_df['voter_name'].isna() == True]

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker


Lastly, we need to handle some of the expected naming trends in the dataset. There are two major ones to focus on: the * which indicates that a vote was an 'after vote' and the -- -- dashes that indicates when the party changes in the roll call.

We want to preserve the after_vote information since that is relevant to our output, but we don't care about the party change dashes so we want to parse and strip these from names that have them.

In [16]:
# Handling * in names

# Add column with True when name contains an *
vote_df['is_after_vote'] = vote_df['voter_name'].str.contains('\*')

# Strip * from all voter_names
vote_df['voter_name'] = vote_df['voter_name'].map(lambda x: x.lstrip('*').rstrip('*').replace('*', ''))

# Print cleaned dataframe
vote_df[vote_df['voter_name'].str.contains('\*') == True]

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker,is_after_vote


In [17]:
# Handle -- -- 
vote_df['voter_name'] = vote_df['voter_name'].map(lambda x: x.lstrip('--').rstrip('--'))

vote_df[vote_df['voter_name'].str.contains('--') == True]

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker,is_after_vote


In [18]:
vote_df

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker,is_after_vote
0,183,1,Finneran,Y,True,False
1,183,1,Fresolo,Y,False,False
2,183,1,"Murphy, K.J.",Y,False,False
3,183,1,Swan,Y,False,False
4,183,1,Asselin,Y,False,False
...,...,...,...,...,...,...
863800,194,88,Xiarhos,Y,False,False
863801,194,88,Field,X,False,False
863802,194,88,MacGregor,Y,False,False
863803,194,88,Sousa,Y,False,False


## Run Matching on Reps

### Test Match votes for a single general court

To test a matching function, I took a subset of the vote dataset and rep dataset based on the first General Court the vote dataset covers: 183.

In [19]:
test_votes_df = vote_df[vote_df['court_number'] == 189]
test_votes_clean_df = test_votes_df.dropna(subset=['voter_name'])

In [20]:
test_votes_clean_df

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker,is_after_vote
524819,189,1,DeLeo,N,True,False
524820,189,1,Ferrante,N,False,False
524821,189,1,Michlewitz,N,False,False
524822,189,1,Vincent,N,False,False
524823,189,1,Arciero,N,False,False
...,...,...,...,...,...,...
614052,189,278,Whipps Lee,Y,False,False
614053,189,278,Ehrlich,X,False,False
614054,189,278,McGonag'e,Y,False,False
614055,189,278,Vega,Y,False,False


In [21]:
test_reps_df = rep_df[rep_df['court_number'] == 189]

In [22]:
test_reps_df

,full_name,member_code,branch,party_affiliation,district,court_year,court_number
1498,Adrian Madaro,ACM1,House,Democrat,1st Suffolk,2015,189
1499,David Paul Linsky,DPL1,House,Democrat,5th Middlesex,2015,189
1500,Marc T. Lombardo,MTL1,House,Republican,22nd Middlesex,2015,189
1501,"James J. Lyons, Jr.",JJL1,House,Republican,18th Essex,2015,189
1502,Timothy R. Madden,TRM1,House,Democrat,"Barnstable, Dukes and Nantucket",2015,189
...,...,...,...,...,...,...,...
1664,Hannah Kane,HEK1,House,Republican,11th Worcester,2015,189
1665,Thomas P. Walsh,TJW1,House,Democrat,12th Essex,2015,189
1666,Gerard Cassidy,G_C2,House,Democrat,9th Plymouth,2015,189
1667,Stephan Hay,S_H1,House,Democrat,3rd Worcester,2015,189


An **enhanced** function `matchName(str, str, fuzzy_threshold=2)` for matching names with improved handling of OCR artifacts and edge cases.

**Key improvements:**
1. **OCR artifact handling**: Cleans names by removing extra spaces, stray periods, and other common OCR errors
2. **Fuzzy matching**: Uses Levenshtein distance to match last names with up to 2 character differences (configurable)
3. **Robust initial matching**: Better handles various initial formats (J.M., J M, JM, etc.) and matches against both first and middle names
4. **Case-insensitive matching**: All comparisons are case-insensitive for better robustness

**Function behavior:**
- Handles comma-separated format: "Last, First M."
- Handles space-separated format: "Last F.M." or "Last F M"  
- Handles single last names
- When initials are present, verifies they match the first letters of the representative's first/middle names in order
- Falls back to fuzzy matching when exact matches fail (helps with OCR errors like "Smtih" vs "Smith")

In [23]:
def clean_name(name):
    """
    Clean a name by removing OCR artifacts and normalizing spacing.
    Handles extra periods, spaces, and other common OCR errors.
    """
    import re
    
    # Remove extra spaces (multiple spaces -> single space)
    name = re.sub(r'\s+', ' ', name)
    
    # Remove periods that aren't followed by a space or end of string (OCR artifacts)
    # But keep periods that are part of initials (followed by space or end)
    name = re.sub(r'\.(?=\S)', '', name)
    
    # Strip leading/trailing whitespace and punctuation
    name = name.strip().strip('.,;-')
    
    return name


def levenshtein_distance(s1, s2):
    """
    Calculate the Levenshtein distance between two strings.
    Returns the minimum number of single-character edits needed.
    """
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            # Cost of insertions, deletions, or substitutions
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]


def fuzzy_match_last_name(last_name, full_name, threshold=2):
    """
    Check if last_name fuzzy matches any word in full_name.
    Returns True if any word in full_name has edit distance <= threshold.
    """
    words = full_name.split()
    for word in words:
        if levenshtein_distance(last_name.lower(), word.lower()) <= threshold:
            return True
    return False


def extract_initials(name_part):
    """
    Extract all initials from a name part.
    Handles formats like "J.M.", "J M", "JM", "J. M.", etc.
    Returns a list of uppercase letters.
    """
    import re
    # Remove periods and extract all uppercase letters or first letters of words
    cleaned = name_part.replace('.', ' ').strip()
    # Get all single capital letters or first letters of multi-letter words
    initials = []
    for part in cleaned.split():
        if part:  # Non-empty
            initials.append(part[0].upper())
    return initials


def check_initials_match(initials, full_name):
    """
    Check if initials match the first letters of names in full_name.
    Handles both "First Middle Last" and "Last, First Middle" formats.
    """
    # Parse full_name to get name components
    if ',' in full_name:
        # Format: "Last, First Middle"
        parts = full_name.split(',')
        if len(parts) >= 2:
            first_middle = parts[1].strip().split()
        else:
            first_middle = []
    else:
        # Format: "First Middle Last" - get all but last word
        words = full_name.split()
        if len(words) > 1:
            first_middle = words[:-1]  # All words except last (which is last name)
        else:
            first_middle = words
    
    # Get first letters from first/middle names
    name_initials = [name[0].upper() for name in first_middle if name]
    
    # Check if provided initials match (in order, but allow partial matches)
    if len(initials) == 0:
        return True
    
    if len(initials) > len(name_initials):
        return False
    
    # Check if initials match the first N name initials
    for i, initial in enumerate(initials):
        if i >= len(name_initials) or initial != name_initials[i]:
            return False
    
    return True


def matchName(name_to_match, name_to_check, fuzzy_threshold=2):
    """
    Enhanced name matching function that handles OCR artifacts and fuzzy matching.
    
    Args:
        name_to_match: Name from vote data (may have OCR errors, often just last name or with initials)
        name_to_check: Full name from representative data
        fuzzy_threshold: Maximum edit distance for fuzzy matching (default=2)
    
    Returns:
        True if names match, False otherwise
    """
    # Clean both names
    name_to_match = clean_name(name_to_match)
    name_to_check = clean_name(name_to_check)
    
    # Handle comma-separated format: "Last, First M."
    if ',' in name_to_match:
        parts = name_to_match.split(',')
        last_name = parts[0].strip()
        first_part = parts[1].strip() if len(parts) > 1 else ""
        
        # Extract initials from first part
        initials = extract_initials(first_part) if first_part else []
        
        # Check if last name matches (with fuzzy matching)
        if last_name.lower() in name_to_check.lower() or fuzzy_match_last_name(last_name, name_to_check, fuzzy_threshold):
            # If no initials, just return True
            if not initials:
                return True
            # Check if initials match
            return check_initials_match(initials, name_to_check)
        
        return False
    
    # Handle space-separated format: "Last F.M." or "Last F M"
    elif ' ' in name_to_match:
        parts = name_to_match.split()
        last_name = parts[0]
        first_part = ' '.join(parts[1:]) if len(parts) > 1 else ""
        
        # Extract initials
        initials = extract_initials(first_part) if first_part else []
        
        # Check if last name matches (with fuzzy matching)
        if last_name.lower() in name_to_check.lower() or fuzzy_match_last_name(last_name, name_to_check, fuzzy_threshold):
            # If no initials, just return True
            if not initials:
                return True
            # Check if initials match
            return check_initials_match(initials, name_to_check)
        
        return False
    
    # Single name (just last name)
    else:
        name_to_match = name_to_match.strip()
        
        # Try exact substring match first
        if name_to_match.lower() in name_to_check.lower():
            return True
        
        # Try fuzzy match
        return fuzzy_match_last_name(name_to_match, name_to_check, fuzzy_threshold)


By looping through all rows of the vote dataset subset and calling the **enhanced** matchName function, we can see how many matches are made. The improved function should handle more edge cases and OCR artifacts. 

In [24]:
verbose = False

# Iterate over all rows of test votes df
for i in test_votes_clean_df.index:

    name = test_votes_clean_df.at[i, 'voter_name']

    # Iterate over all rep names in subset
    for j in test_reps_df.index:
        check_name = test_reps_df.at[j, 'full_name']

        # Check if name is found in name to check
        if verbose: print(f'{i}: Checking {name} against {check_name}')
        if matchName(name,check_name):
            member_code = test_reps_df.at[j, 'member_code']
            if verbose: print(f'Found match!! Member Code: {member_code}')
            test_votes_clean_df.at[i, 'member_code'] = member_code
            break

In [25]:
test_votes_clean_df[test_votes_clean_df['member_code'].isna() == True]['voter_name'].unique()

array(['Whipps Lee', 'BerthiaWne', 'Die hl', 'Cantwe ll', "D ' Emilia",
       'Garnpbell', "ait'er", "O ' Connell", 'Petrol a ti'], dtype=object)

### Batch process matching across all General Court Years

Now that we've seen a pretty good result (still will need manual checking, since it appears that 3-5 names per GC are not coming up in the rep dataset), we can run it over all the GCs.

A function ```matchAllNames(pd.DataFrame, pd.DataFrame)``` is defined that iterates over the range of all GCs we are working with. It takes the input of dataframes containing all votes and all reps and does the same subsetting process for each GC. The matching function is called for each row and an output DataFrame containing the matched values is concatenated over the iterations.

At the end of the function, some summary statistics of the matching output are printed for clarity.

In [26]:
%pip install --upgrade jupyter ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [29]:
def matchAllNames(all_votes_df, all_reps_df, verbose=False):
    # Silence SettingsWithCopyWarning for this run
    pd.options.mode.chained_assignment = None

    # Track matches that failed
    failed_matches = 0

    # Create a DataFrame to return
    out_cols = all_votes_df.columns.to_list()
    out_cols.append('member_code')
    out_votes_df = pd.DataFrame(columns=out_cols)

    # Iterate over all General Courts contained in the votes data
    for gc in range(183, 195):
        print(f'Matching reps and votes from General Court #{gc}')

        # Get the votes and representatives for current General Court
        gc_votes_df = all_votes_df[vote_df['court_number'] == gc]
        gc_reps_df = all_reps_df[rep_df['court_number'] == gc]

        # Iterate over all rows of test votes df - use tqdm to show progress of current GC matching
        for i in tqdm(gc_votes_df.index):

            name = gc_votes_df.at[i, 'voter_name']

            # Iterate over all rep names in subset
            for j in gc_reps_df.index:
                check_name = gc_reps_df.at[j, 'full_name']

                # Check if name is found in name to check
                if verbose: print(f'{i}: Checking {name} against {check_name}')
                if matchName(name,check_name):
                    member_code = gc_reps_df.at[j, 'member_code']
                    if verbose: print(f'Found match!! Member Code: {member_code}')
                    gc_votes_df.at[i, 'member_code'] = member_code

                    # Break once a match is found since a member should not match more than once
                    break

        # Aggregate number of failed matches
        failed_matches += len(gc_votes_df[gc_votes_df['member_code'].isna() == True])

        # After matching for this GC, concat output to out_votes_df
        out_votes_df = pd.concat([out_votes_df, gc_votes_df])

    # After matching all General Courts
    print(f'Percentage of failed matches: {failed_matches} / {len(out_votes_df)} = {(failed_matches / len(out_votes_df)) * 100}% failure rate')

    return out_votes_df

In [30]:
matches_votes_df = matchAllNames(vote_df, rep_df)

Matching reps and votes from General Court #183


  0%|          | 0/76083 [00:00<?, ?it/s]

Matching reps and votes from General Court #184


  0%|          | 0/133015 [00:00<?, ?it/s]

Matching reps and votes from General Court #185


  0%|          | 0/96088 [00:00<?, ?it/s]

... (15 outputs left out)

  0%|          | 0/32299 [00:00<?, ?it/s]

Matching reps and votes from General Court #194


  0%|          | 0/14139 [00:00<?, ?it/s]

Percentage of failed matches: 5409 / 863805 = 0.6261829926893223% failure rate


After matching is done, we see a pretty good result of about only a 10% failure rate. I know from my knowledge of the data that a majority of this comes from the 2009 - 2017 section of the data so this is something we will need to deal with directly.

In [33]:
matches_votes_df

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker,is_after_vote,member_code
0,183,1,Finneran,Y,True,False,TMF1
1,183,1,Fresolo,Y,False,False,JPF1
2,183,1,"Murphy, K.J.",Y,False,False,KJM1
3,183,1,Swan,Y,False,False,B S1
4,183,1,Asselin,Y,False,False,CPA1
...,...,...,...,...,...,...,...
863800,194,88,Xiarhos,Y,False,False,SGX1
863801,194,88,Field,X,False,False,CAF1
863802,194,88,MacGregor,Y,False,False,WFM1
863803,194,88,Sousa,Y,False,False,PSS1


In [ ]:
failed_match_names = matches_votes_df[matches_votes_df['member_code'].isna() == True]['voter_name'].unique().tolist()
failed_match_names

['Galvin',
 'Petrol a ti',
 "L ' Italien",
 'Alle n',
 "L ' Ita'ien",
 "O ' Day",
 'Walsh, M J',
 'Walsh , M J',
 "L' Italien",
 "D ' Amico",
 't-rurphy, J',
 't-rurphy, KJ',
 "H'wnason",
 'Petro la ti',
 "D ' Arnico",
 "O ' Flaherty",
 'Dwye r',
 'Mci-rurtry',
 'Orisco',
 "L' Ita'ien",
 'Demps ey',
 'Jone s',
 'Garnpbell',
 'Michlewi tz',
 'Brownsberger',
 't-rurphy, K',
 't-rurphy, C',
 'Die hl',
 'GampbeU',
 "D ' Emilia",
 'Hwnason',
 'Reinste in',
 'Bradle y',
 "ait'er",
 'Pignate lli',
 'OJinane',
 'calte r',
 'deMace do',
 'Whipps Lee',
 'BerthiaWne',
 'Cantwe ll',
 "O ' Connell",
 'Berthiawne',
 'saDcbez',
 'Berthiawne ',
 'SaDcbez',
 'Rogers , o',
 'Livings tone',
 'Tyle r',
 'saDcbez ',
 'Vincent',
 'Nangle',
 'Poirier',
 'Naughton',
 'Hecht',
 'Crocker',
 'Petrolati',
 'Provost',
 'Kafka',
 'Hunt, R.',
 'Scaccia',
 'Speliotis',
 'Tosado',
 'Fluker Oakley']

In [61]:
matches_votes_df

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker,is_after_vote,member_code
0,183,1,Finneran,Y,True,False,TMF1
1,183,1,Fresolo,Y,False,False,JPF1
2,183,1,"Murphy, K.J.",Y,False,False,KJM1
3,183,1,Swan,Y,False,False,B S1
4,183,1,Asselin,Y,False,False,CPA1
...,...,...,...,...,...,...,...
863800,194,88,Xiarhos,Y,False,False,SGX1
863801,194,88,Field,X,False,False,CAF1
863802,194,88,MacGregor,Y,False,False,WFM1
863803,194,88,Sousa,Y,False,False,PSS1


In [62]:
matches_votes_df[matches_votes_df['voter_name'] == "D ' Arnico"]

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker,is_after_vote,member_code
328448,186,23,D ' Arnico,Y,False,False,NaN


### Manual Corrections for Failed Matches

Some voter names failed to match due to severe OCR artifacts or unusual formatting that even fuzzy matching couldn't handle. The dictionary below maps these problematic voter names to their correct member codes, identified through manual review. While ideally this would have been fully automated, the OCR process lead to alot of inconsitencies that were more annoying to deal with programmatically than it was to correct 50 or so names that couldn't be matched.

In [57]:
failed_match_corrections = {
    'Galvin': 'WCG1',
    'Petrol a ti': 'TMP1',
    "L ' Italien": 'BAL1',
    'Alle n' : 'WMA1',
    "L ' Ita'ien": 'BAL1',
    "O ' Day": 'JJO1',
    'Walsh, M J': 'MJW1',
    'Walsh , M J': 'MJW1',
    "L' Italien": 'BAL1',
    "D ' Amico": 'SJD1',
    't-rurphy, J': 'JMM1',
    't-rurphy, KJ': 'KJM1',
    "H'wnason": 'DFH1', 
    'Petro la ti': 'TMP1',
    "D ' Arnico": 'SJD1', 
    "O ' Flaherty": 'ELO1', 
    'Dwye r': 'JJD1', 
    'Mci-rurtry': 'P_M1',
    'Orisco': 'JRD1',
    "L' Ita'ien": 'BAL1' ,
    'Demps ey': 'BSD1',
    'Jone s': 'BHJ1',
    'Garnpbell': 'LDC1',
    'Michlewi tz': 'AMM1',
    'Brownsberger': 'WNB1',
    't-rurphy, K': 'KJM1',
    't-rurphy, C': 'CAM1',
    'Die hl': 'J_D1',
    'GampbeU': 'LDC1',
    "D ' Emilia": 'ALD1',
    'Hwnason': 'DFH1', 
    'Reinste in': 'K_R1',
    'Bradle y': 'GJB1',
    "ait'er": 'JSC1',
    'Pignate lli': 'WSP1',
    'OJinane': 'D_C1',
    'calte r': 'TJC1',
    'deMace do': 'VMD1',
    'Whipps Lee': 'SLG1',
    'BerthiaWne':  'DRB1',
    'Cantwe ll': 'JMC2',
    "O ' Connell": 'SLO1',  
    'Berthiawne': 'DRB1',
    'saDcbez': 'J_S1',
    'Berthiawne ': 'DRB1',
    'SaDcbez': 'J_S1',
    'Rogers , o': 'DMR1',
    'Livings tone': 'J_L1',
    'Tyle r': 'C_T1',
    'saDcbez ': 'J_S1',
    'Vincent': 'r_v1',
    'Nangle': 'DMN1',
    'Poirier': 'EAP1',
    'Naughton': 'HPN1', 
    'Hecht': 'J_H1',
    'Crocker': 'WLC1',
    'Petrolati': 'TMP1', 
    'Provost': 'D_P1', 
    'Kafka': 'LLK1',  
    'Hunt, R.': 'R_H1', 
    'Scaccia': 'AMS1',
    'Speliotis': 'TCS1',
    'Tosado': 'JFT1',
    'Fluker Oakley': 'BFO1'
}

In [64]:
# Apply manual corrections for failed matches
# For rows where member_code is NaN, check if voter_name is in our corrections dictionary
# and fill with the correct member_code

# Create a mask for rows with NaN member_code
nan_mask = matches_votes_df['member_code'].isna()

# For those rows, map voter_name to the corrections dictionary and fill NaN values
matches_votes_df.loc[nan_mask, 'member_code'] = matches_votes_df.loc[nan_mask, 'voter_name'].map(failed_match_corrections)

# Show how many corrections were applied
corrections_applied = matches_votes_df.loc[nan_mask, 'member_code'].notna().sum()
print(f"Applied {corrections_applied} manual corrections from the failed_match_corrections dictionary")

Applied 5409 manual corrections from the failed_match_corrections dictionary


In [65]:
# Verify corrections were applied - show a few examples
sample_corrections = ["D ' Arnico", "Petrol a ti", "t-rurphy, J", "Michlewi tz"]
print("Sample of corrected voter names:\n")
for name in sample_corrections:
    sample = matches_votes_df[matches_votes_df['voter_name'] == name][['voter_name', 'member_code']].head(3)
    if not sample.empty:
        print(f"{name}:")
        print(sample)
        print()


Sample of corrected voter names:

D ' Arnico:
        voter_name member_code
328448  D ' Arnico        SJD1

Petrol a ti:
         voter_name member_code
305420  Petrol a ti        TMP1
307169  Petrol a ti        TMP1
310667  Petrol a ti        TMP1

t-rurphy, J:
         voter_name member_code
315083  t-rurphy, J        JMM1
387307  t-rurphy, J        JMM1
390153  t-rurphy, J        JMM1

Michlewi tz:
         voter_name member_code
382295  Michlewi tz        AMM1
405620  Michlewi tz        AMM1



In [66]:
# Check remaining failed matches after applying corrections
remaining_failed = matches_votes_df[matches_votes_df['member_code'].isna()]
total_rows = len(matches_votes_df)
failed_count = len(remaining_failed)

print(f"Remaining failed matches: {failed_count} / {total_rows} = {(failed_count / total_rows) * 100:.4f}% failure rate")
print(f"\nUnique voter names that still failed to match:")

failed_match_names = remaining_failed['voter_name'].unique().tolist()
failed_match_names

Remaining failed matches: 0 / 863805 = 0.0000% failure rate

Unique voter names that still failed to match:


[]

## Save output to csv

In [67]:
matches_votes_df

,court_number,rollcall_count_for_year,voter_name,vote_value,is_speaker,is_after_vote,member_code
0,183,1,Finneran,Y,True,False,TMF1
1,183,1,Fresolo,Y,False,False,JPF1
2,183,1,"Murphy, K.J.",Y,False,False,KJM1
3,183,1,Swan,Y,False,False,B S1
4,183,1,Asselin,Y,False,False,CPA1
...,...,...,...,...,...,...,...
863800,194,88,Xiarhos,Y,False,False,SGX1
863801,194,88,Field,X,False,False,CAF1
863802,194,88,MacGregor,Y,False,False,WFM1
863803,194,88,Sousa,Y,False,False,PSS1


In [69]:
final_vote_df = matches_votes_df[['court_number', 'rollcall_count_for_year', 'member_code', 'vote_value', 'is_speaker', 'is_after_vote']]

In [70]:
final_vote_df

,court_number,rollcall_count_for_year,member_code,vote_value,is_speaker,is_after_vote
0,183,1,TMF1,Y,True,False
1,183,1,JPF1,Y,False,False
2,183,1,KJM1,Y,False,False
3,183,1,B S1,Y,False,False
4,183,1,CPA1,Y,False,False
...,...,...,...,...,...,...
863800,194,88,SGX1,Y,False,False
863801,194,88,CAF1,X,False,False
863802,194,88,WFM1,Y,False,False
863803,194,88,PSS1,Y,False,False


Lastly, we can write the output to a CSV file that will be saved to the project Google Drive.

In [72]:
final_vote_df.to_csv('../data/parsedcsv/hjrollcallvotes_final.csv')

In [ ]:
test_read_df = pd.read_csv('../../data/parsedcsv/hjrollcallvotes_final.csv')
test_read_df